In [ ]:
from option_chain_downloader import OptionChainDownloader
from option_finder import *
from option_data_plotter import *
import matplotlib.pyplot as plt

In [ ]:
try:
    logger
except NameError:
    logger = get_rotating_logger("jupyter", f'logs/leaps_latest.log')

In [ ]:
chain_dir = 'chain'
quotes_dir = 'quotes'
data_dir = 'data'
cookie_file = 'cookie.txt'
ocd = OptionChainDownloader(chain_dir, quotes_dir, cookie_file, logger, strikes='ALL')
self = OptionFinder(logger, chain_dir=chain_dir, report_dir=data_dir)

In [ ]:
#symlist = ['GLD', 'QQQ'] # TLT GLD IBIT'.split(' ')
symlist = 'QQQ SPY TLT DIA IBIT ETHA GLD CRCL NVDA MSFT GOOGL AAPL'.split(' ')
#symlist = 'TSM NVDA AAPL MSFT GOOGL AMZN META TSLA'.split(' ')
#symlist = 'QQQ SPY TLT DIA IBIT GLD'.split(' ')

## Refresh data here

In [ ]:
_t0 = time.time()
_n = ocd.download_option_chain(symlist, batch_size=5, rps=5)
print(_n, 'files downloaded', int(time.time() - _t0), 'seconds')

In [ ]:
self.get_quote_df(symlist)
_df = self.build_option_df(symlist)
df_cp = self.concat_put_call_options(_df)
df_cp = df_cp[df_cp.dte >= 90].copy()
df_cp['pctTheta'] = df_cp.Theta*100/df_cp.mid
px.bar(check_data_age(_df), y=['load_age', 'quote_age'], barmode='group', title=f"Data Ages", width=len(symlist)*80, height=300).show()
df_cp.loc[:, ['dte', 'expDt']].groupby('dte').first().tail(20).T

In [ ]:
pd.cut(df_cp['dte'], bins=[0, 7, 56, 91, 182, 364, np.inf], labels=['1wk', '8wk', '13wk', '6mo', '1yr', '>1yr'])

In [ ]:
_dfoi = df_cp[df_cp.OpenInterest > 0].pivot(columns=['type'], index=['symbol', 'dte', 'strike'], values=['OpenInterest'])
_dfoi.columns = [c[1] for c in _dfoi.columns]
_dfoi = _dfoi.reset_index().drop(columns=['dte', 'strike'])
_g = _dfoi.groupby('symbol')
pd.concat([_g.quantile(0.25).astype(int), _g.quantile(0.5).astype(int), _g.mean().astype(int), _g.max().astype(int)], axis=1)

In [ ]:
_df = add_moneyness_columns(df_cp)
_df[_df.OpenInterest > 0].pivot_table(columns=['type'], index=['symbol', 'dte', 'cluster'], values=['OpenInterest', 'Volume'], observed=True)

### Puts/calls ratios for all strikes

In [ ]:
plot_put_call_ratios(df_cp[df_cp.symbol != 'DIA'], [180, 200])

In [ ]:
_df = plot_leverage_pct_theta(df_cp, delta_lb=0.5, overpaid_ub=0.05, price_lb=5, spread_ub=10, leverage_lb=2, openinterest_lb=10)

In [ ]:
_df = plot_leverage_overpaid(df_cp, delta_lb=0.5, overpaid_ub=0.05, price_lb=5, spread_ub=10, leverage_lb=2, openinterest_lb=10)

In [ ]:
_symbol = 'QQQ'
dte_filter = (df_cp.symbol == _symbol) & (df_cp.dte >= 90) & (df_cp.dte <= 360)
df_moneyness = add_moneyness_columns(df_cp[dte_filter].copy(), atm_offset=0.01, otm_offset=0.1)
_df = plot_put_call_ratios_by_moneyness(df_moneyness)

In [ ]:
_df[_df.symbol=='GLD']

In [ ]:
_symbol = 'GLD'
_df2 = df_cp[(df_cp.symbol==_symbol) & (df_cp.dte == 108) & (df_cp.strike >= 200) & (df_cp.pctTheta >= -0.1)]
plot_leverage_overpaid(_df2, delta_lb=0.5, overpaid_ub=0.05, price_lb=1, spread_ub=10, leverage_lb=3, openinterest_lb=100).loc[_symbol].sort_values(by='OpenInterest').tail(60)

In [ ]:
plot_option_details_by_dte(df_cp, 'TLT', 'pctTheta', delta_lb=0.5, delta_ub=0.9, nr=4, nc=3)

In [ ]:
df_cp.columns

In [ ]:
plot_option_details_by_strike(df_cp, 'TLT', 'pctTheta', [0.8, 0.95])

### The End